# Downsampling Eaglei Data

The goal of this notebook is to downsample the eaglei data for each year to a 6-hour cadence, replacing the number of customers without power with the 6-hour mean

This will also combine the data for all years into a single file

Then the data can be exported into a csv or parquet file

In [41]:
import pandas as pd
import numpy as np
import os

In [ ]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_')]

#Create an empty data frame
outages = pd.DataFrame()

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Drop the county and state variables to facilitate up/downsampling (these could be merged back in later if needed)
    df.drop(['county', 'state'], axis=1, inplace=True)

    #Convert the run_start_time variable into datetime
    df['run_start_time'] = pd.to_datetime(df['run_start_time'])
    df.set_index('run_start_time', inplace=True)

    #Group the data by fips_code. Add new values of run_start_time every 15 minutes.
    # Forward fill values of customers_out with 0
    # Then ungroup the data
    df = df.groupby('fips_code').resample('15min').asfreq().fillna(0).reset_index(level=0, drop=True)

    #In the fips_code variable, replace 0 with NA
    df['fips_code'] = df['fips_code'].replace(0, np.nan)

    #Forward fill the fips_code with the most recent value
    df['fips_code'] = df['fips_code'].ffill()

    #Group the data by fips_code. Then downsample the data to every 6 hours, replacing customers_out with the mean and then ungroup the data
    df = df.groupby('fips_code').resample('6h').mean().reset_index(level=0, drop=True)


    #Concatenate df with outages
    outages = pd.concat([outages, df])

#For exporting and future merging, move the datetime back to a "regular" variable and reset the index
outages['datetime'] = pd.to_datetime(outages.index)
outages.reset_index(drop=True, inplace=True)


In [35]:
#Export outages to a csv
outages.to_csv('../Data/eaglei_data/eaglei_outages.csv')

In [36]:
#Export outages to a parquet
outages.to_parquet('../Data/eaglei_data/eaglei_outages.parquet')

Next, we can merge the outages data with the additional county variables from the Counties_All file

In [ ]:
#Add a YEAR variable from datetime
outages['YEAR'] = outages['datetime'].dt.year

#Load ../Data/Counties_All.csv
counties = pd.read_csv('../Data/Counties_All.csv')

# Merge outages and counties based on the YEAR variable and the fips_code/FIPS variables
outages_merged = outages.merge(counties, left_on=['YEAR', 'fips_code'], right_on=['YEAR', 'FIPS'])

In [ ]:
#Export outages_merged to a csv
#Note that this will take about 6 minutes and generate a ~15 GB file, so proceed with caution
outages_merged.to_csv('../Data/eaglei_data/eaglei_outages_with_county_info.csv')

In [55]:
#Export outages_merged to a parquet
outages_merged.to_parquet('../Data/eaglei_data/eaglei_outages_with_county_info.parquet')